In [4]:
import tensorflow as tf
from PIL import Image
from pathlib import Path


In [5]:
def augument(image, masks):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        masks = tf.image.flip_left_right(masks)

    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        masks = tf.image.flip_up_down(masks)
    k = tf.random.uniform(
        shape=[],
        minval=0,
        maxval= 4,
        dtype=tf.int32
    )
    image = tf.image.rot90(image,k)
    masks = tf.image.rot90(masks, k)

    image = tf.image.random_brightness(
        image,
        max_delta = 0.1
    )
    image = tf.clip_by_value(
        image, 
        0.0,
        1.0
    )
    return image, masks

In [6]:
datasetPath = Path("../dataset/data_wound_seg_256")

train_images_path = sorted(
    (datasetPath / "train_images").glob("*.png")
)

train_masks_path = sorted(
    (datasetPath / "train_masks").glob("*.png")
)

train_images_path = [str(p) for p in train_images_path]
train_masks_path = [str(p) for p in train_masks_path]



test_images_path = sorted(
    (datasetPath / "test_images").glob("*.png")
)

test_masks_path = sorted(
    (datasetPath / "test_masks").glob("*.png")
)

test_images_path = [str(p) for p in test_images_path]
test_masks_path = [str(p) for p in test_masks_path]

In [7]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_images_path, train_masks_path)
)

2026-08-23 00:42:28.925886: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-23 00:42:28.929453: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-23 00:42:28.931386: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [8]:
def load_images_masks(image_path, mask_path):

    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.convert_image_dtype(mask, tf.float32)

    return image, mask

In [9]:
BUFFER_SIZE = len(train_images_path)
BATCH_SIZE = 2

In [10]:
train_dataset = train_dataset.map(
    load_images_masks,
    num_parallel_calls=tf.data.AUTOTUNE
    )
train_dataset = train_dataset.map(
    augument,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.shuffle(
    buffer_size=BUFFER_SIZE
)
train_dataset = train_dataset.batch(
    batch_size=BATCH_SIZE
)
train_dataset= train_dataset.prefetch(
    tf.data.AUTOTUNE
)

In [11]:
test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_images_path, test_masks_path)
)

test_dataset = test_dataset.map(
    load_images_masks,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_dataset = test_dataset.batch(
    BATCH_SIZE
)

test_dataset = test_dataset.prefetch(
    tf.data.AUTOTUNE
)

In [12]:
model = tf.keras.models.load_model(
    "../models/unet_bce_dice_best.keras",
    compile=False
)

In [13]:
def dice_loss(y_true, y_pred, smooth=1e-6):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        tf.reduce_sum(y_true)
        + tf.reduce_sum(y_pred)
        + smooth
    )

    return 1.0 - dice


def bce_dice_loss(y_true, y_pred):

    bce = tf.keras.losses.binary_crossentropy(
        y_true,
        y_pred
    )

    bce = tf.reduce_mean(bce)

    dice = dice_loss(
        y_true,
        y_pred
    )

    return bce + dice

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=bce_dice_loss,
    metrics=["accuracy"]
)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "../models/unet_dataset1_bce_dice_aug_best.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min"
)

In [16]:
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10,
    callbacks=[checkpoint]
)

Epoch 1/10


2026-08-23 00:42:59.123150: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
2026-08-23 00:43:02.035203: I external/local_xla/xla/service/service.cc:168] XLA service 0x799040fb02d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-08-23 00:43:02.035227: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-08-23 00:43:02.043032: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787425982.107711   51223 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-08-23 00:43:02.281895: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.14GiB with freed_by_count=0. The caller

1104/1104 [==============================] - 192s 160ms/step - loss: 0.2527 - accuracy: 0.9850 - val_loss: 0.2030 - val_accuracy: 0.9961
Epoch 2/10
1104/1104 [==============================] - 186s 167ms/step - loss: 0.2219 - accuracy: 0.9868 - val_loss: 0.1946 - val_accuracy: 0.9963
Epoch 3/10
1104/1104 [==============================] - 186s 168ms/step - loss: 0.2202 - accuracy: 0.9868 - val_loss: 0.1958 - val_accuracy: 0.9963
Epoch 4/10
1104/1104 [==============================] - 187s 169ms/step - loss: 0.2122 - accuracy: 0.9870 - val_loss: 0.1961 - val_accuracy: 0.9962
Epoch 5/10
1104/1104 [==============================] - 187s 169ms/step - loss: 0.2051 - accuracy: 0.9876 - val_loss: 0.1972 - val_accuracy: 0.9962
Epoch 6/10
1104/1104 [==============================] - 188s 170ms/step - loss: 0.2065 - accuracy: 0.9877 - val_loss: 0.1923 - val_accuracy: 0.9963
Epoch 7/10
1104/1104 [==============================] - 188s 170ms/step - loss: 0.2018 - accuracy: 0.9878 - val_loss: 0.195

In [17]:
import tensorflow as tf
import numpy as np

model = tf.keras.models.load_model(
    "../models/unet_bce_dice_aug_best.keras",
    compile=False
)

In [18]:
dice_scores = []
iou_scores = []

for images, masks in test_dataset:

    predictions = model.predict(
        images,
        verbose=0
    )

    for i in range(images.shape[0]):

        gt = masks[i].numpy().squeeze() > 0.5
        pred = predictions[i].squeeze() > 0.5

        intersection = np.logical_and(
            gt,
            pred
        ).sum()

        union = np.logical_or(
            gt,
            pred
        ).sum()

        dice = (2 * intersection) / (
            gt.sum() + pred.sum() + 1e-7
        )

        iou = intersection / (
            union + 1e-7
        )

        dice_scores.append(dice)
        iou_scores.append(iou)

print("Test images:", len(dice_scores))
print("Mean Dice:", np.mean(dice_scores))
print("Mean IoU:", np.mean(iou_scores))

Test images: 552
Mean Dice: 0.7802093241675617
Mean IoU: 0.6906025329518821


In [19]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print(gpus)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [20]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print(gpus)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [21]:
print(tf.config.experimental.get_memory_growth(gpus[0]))

True
